In [1]:
import os
import numpy as np
import pandas as pd
import datetime as dt
# from datetime import date, datetime
from dateutil.relativedelta import relativedelta

import yfinance as yf
import pandas_datareader as pdr

### Download ETF Prices

In [2]:
DATA_DIR   = './data/'
START_DATE = '2004-12-01' # extra month so we can calculate return in Jan 2005
END_DATE   = '2026-04-30'

# choose a few ETFs to represent assets a retail investor can access
TICKERS = ['SPY',  # US large cap
           'IWM',  # US small cap
           'QQQ',  # US tech heavy
           'EFA',  # Developed international
           'EEM',  # Emerging markets
           'TLT',  # Long-term Treasuries
           'TIP',  # TIPS (Treasury Inflation-Protected securities)
           'LQD',  # Investment grade corporate bonds
           'GLD',  # Gold
           'VNQ',  # US REITs (Real Estate Investment Trusts)
           ]

In [3]:
print('Downloading daily adjusted close prices...')
daily_prices = yf.download(tickers     = TICKERS,
                           start       = START_DATE,
                           end         = END_DATE,
                           auto_adjust = True, # adjusts for splits and dividends
                           progress    = True,
                           )['Close']

monthly_prices = daily_prices.resample('ME').last() # resample for month-end

print(f'\nDate range:     {monthly_prices.index[0].date()} to {monthly_prices.index[-1].date()}')
print(f'Number of months: {len(monthly_prices)}')
print(f'\nFirst available date per ticker:')
print(monthly_prices.apply(lambda col: col.first_valid_index()).to_string())

monthly_prices.head()

[*********************100%***********************]  10 of 10 completed



Date range:     2004-12-31 to 2026-04-30
Number of months: 257

First available date per ticker:
Ticker
EEM   2004-12-31
EFA   2004-12-31
GLD   2004-12-31
IWM   2004-12-31
LQD   2004-12-31
QQQ   2004-12-31
SPY   2004-12-31
TIP   2004-12-31
TLT   2004-12-31
VNQ   2004-12-31


Ticker,EEM,EFA,GLD,IWM,LQD,QQQ,SPY,TIP,TLT,VNQ
Date,,,,,,,,,,
2004-12-31,14.639680,28.706861,43.799999,48.706833,47.044949,33.973446,81.559227,54.419136,44.138405,22.614822
2005-01-31,14.562080,28.160488,42.220001,46.728470,47.622860,31.828817,79.730598,54.424267,45.713547,20.679270
2005-02-28,15.972013,29.226370,43.529999,47.495770,47.176991,31.675646,81.397263,54.136879,45.040573,21.335114
2005-03-31,14.708583,28.459665,42.820000,46.149361,46.581516,31.122458,79.908287,54.255367,44.835857,20.970814
2005-04-30,14.525085,27.999266,43.349998,43.538712,47.336906,29.769316,78.411201,55.219769,46.568794,22.180828


In [5]:
output_path = os.path.join(DATA_DIR, 'monthly-etf-prices.parquet')
monthly_prices.to_parquet(output_path)
print(f'\nSaved monthly ETF price data to {output_path}')

output_path = os.path.join(DATA_DIR, 'daily-etf-prices.parquet')
daily_prices.to_parquet(output_path)
print(f'\nSaved daily ETF price data to {output_path}')


Saved monthly ETF price data to ./data/monthly-etf-prices.parquet

Saved daily ETF price data to ./data/daily-etf-prices.parquet


### Get Risk-Free-Rate Data
Pull risk-free-rate from the Fama-French monthly factor dataset

In [10]:
# pull the fama-french 3 factor dataset
ff_start_date = dt.datetime.strptime(START_DATE, '%Y-%m-%d').date() + relativedelta(months=1)
ff_end_date   = dt.datetime.strptime(END_DATE, '%Y-%m-%d').date()

ff_factors = pdr.get_data_famafrench('F-F_Research_Data_Factors', start=ff_start_date, end=ff_end_date)[0]
rfr        = pd.DataFrame(ff_factors['RF'] / 100.)  # convert from percent to decimal

# convert index to datetime and add the month-end day to match the yfinance data
rfr.index = pd.to_datetime(rfr.index.astype(str), format='%Y-%m') + pd.offsets.MonthEnd(0)

rfr.head()

/tmp/ipykernel_57333/2795651571.py:5: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_factors = pdr.get_data_famafrench('F-F_Research_Data_Factors', start=ff_start_date, end=ff_end_date)[0]
/tmp/ipykernel_57333/2795651571.py:5: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_factors = pdr.get_data_famafrench('F-F_Research_Data_Factors', start=ff_start_date, end=ff_end_date)[0]


,RF
Date,
2005-01-31,0.0016
2005-02-28,0.0016
2005-03-31,0.0021
2005-04-30,0.0021
2005-05-31,0.0024


In [11]:
output_path = os.path.join(DATA_DIR, 'monthly-rfr.parquet')
rfr.to_parquet(output_path)
print(f'\nSaved risk-free-rate data to {output_path}')


Saved risk-free-rate data to ./data/monthly-rfr.parquet
